# 10 Silver CarePlan Clean

## Purpose

This notebook creates the Silver CarePlan table from raw FHIR CarePlan resources.

## What We Are Doing

We will:
1. Read `healthcare_catalog.bronze.careplan_raw`
2. Extract care plan fields
3. Flatten selected nested FHIR fields
4. Clean patient and encounter IDs
5. Convert care plan dates
6. Save the clean table into the Silver layer

## Why We Are Doing This

CarePlan data is important for:
- chronic disease management
- care coordination analytics
- treatment pathway analysis
- patient risk scoring
- population health analytics

## Expected Final Output

A clean Delta table:

`healthcare_catalog.silver.careplan_clean`

## Step 1 — Import PySpark Functions

### What We Are Doing
We are importing Spark SQL functions.

### Why We Are Doing This
We need Spark functions to extract nested FHIR fields and clean references.

### Expected Output
PySpark functions available.

In [0]:
from pyspark.sql.functions import *

## Step 2 — Read Bronze CarePlan Table

### What We Are Doing
We are reading the raw CarePlan table from the Bronze layer.

### Why We Are Doing This
Bronze contains raw nested FHIR CarePlan resources.

### Expected Output
A DataFrame named `careplan_raw_df`.

In [0]:
careplan_raw_df = spark.table(
    "healthcare_catalog.bronze.careplan_raw"
)

print("Bronze careplan_raw table loaded successfully.")

Bronze careplan_raw table loaded successfully.


## Step 3 — Inspect Raw CarePlan Schema

### What We Are Doing
We are printing the CarePlan schema.

### Why We Are Doing This
FHIR CarePlan resources are nested. We need to confirm exact fields before flattening.

### Expected Output
Schema fields such as:
- resource.id
- resource.subject.reference
- resource.encounter.reference
- resource.status
- resource.intent
- resource.category
- resource.activity
- resource.period.start
- resource.period.end
- resource.addresses

In [0]:
careplan_raw_df.printSchema()

root
 |-- fullUrl: string (nullable = true)
 |-- resourceType: string (nullable = true)
 |-- resource: struct (nullable = true)
 |    |-- abatementDateTime: string (nullable = true)
 |    |-- active: boolean (nullable = true)
 |    |-- activity: array (nullable = true)
 |    |    |-- element: struct (containsNull = true)
 |    |    |    |-- detail: struct (nullable = true)
 |    |    |    |    |-- code: struct (nullable = true)
 |    |    |    |    |    |-- coding: array (nullable = true)
 |    |    |    |    |    |    |-- element: struct (containsNull = true)
 |    |    |    |    |    |    |    |-- code: string (nullable = true)
 |    |    |    |    |    |    |    |-- display: string (nullable = true)
 |    |    |    |    |    |    |    |-- system: string (nullable = true)
 |    |    |    |    |    |-- text: string (nullable = true)
 |    |    |    |    |-- location: struct (nullable = true)
 |    |    |    |    |    |-- display: string (nullable = true)
 |    |    |    |    |-- statu

## Step 4 — Extract Clean CarePlan Columns

### What We Are Doing
We are extracting important care plan fields from the nested FHIR resource.

### Why We Are Doing This
Analytics-ready tables need flat columns instead of nested JSON.

### Expected Output
A DataFrame named `careplan_clean_df`.

In [0]:
careplan_clean_df = careplan_raw_df.select(

    col("resource.id").alias("careplan_id"),

    col("resource.subject.reference").alias("patient_reference"),

    col("resource.encounter.reference").alias("encounter_reference"),

    col("resource.status").alias("careplan_status"),

    col("resource.intent").alias("careplan_intent"),

    col("resource.category")[0].alias("careplan_category"),

    col("resource.activity")[0]["detail"]["code"]["text"].alias("activity_description"),

    col("resource.activity")[0]["detail"]["status"].alias("activity_status"),

    col("resource.addresses")[0]["reference"].alias("reason_reference"),

    col("resource.period.start").alias("start_datetime"),

    col("resource.period.end").alias("end_datetime")
)

print("CarePlan clean DataFrame created successfully.")

CarePlan clean DataFrame created successfully.


## Step 5 — Convert CarePlan Dates

### What We Are Doing
We are converting start and end dates into Spark timestamp format.

### Why We Are Doing This
Timestamp format supports:
- longitudinal care analysis
- care plan duration calculation
- patient timeline analytics

### Expected Output
`start_datetime` and `end_datetime` become timestamp columns.

In [0]:
careplan_clean_df = careplan_clean_df.withColumn(
    "start_datetime",
    to_timestamp(col("start_datetime"))
)

careplan_clean_df = careplan_clean_df.withColumn(
    "end_datetime",
    to_timestamp(col("end_datetime"))
)

print("CarePlan timestamps converted successfully.")

CarePlan timestamps converted successfully.


## Step 6 — Extract Clean Patient, Encounter, and Reason IDs

### What We Are Doing
We are removing `urn:uuid:` from FHIR reference fields.

### Why We Are Doing This
Clean IDs are required for joining CarePlan with Patient, Encounter, and Condition tables.

### Expected Output
New columns:
- patient_id
- encounter_id
- reason_condition_id

In [0]:
careplan_clean_df = careplan_clean_df.withColumn(
    "patient_id",
    regexp_extract(col("patient_reference"), r"urn:uuid:(.*)", 1)
)

careplan_clean_df = careplan_clean_df.withColumn(
    "encounter_id",
    regexp_extract(col("encounter_reference"), r"urn:uuid:(.*)", 1)
)

careplan_clean_df = careplan_clean_df.withColumn(
    "reason_condition_id",
    regexp_extract(col("reason_reference"), r"urn:uuid:(.*)", 1)
)

print("CarePlan IDs extracted successfully.")

CarePlan IDs extracted successfully.


## Step 7 — Calculate CarePlan Duration

### What We Are Doing
We are calculating care plan duration in days.

### Why We Are Doing This
Duration can become a useful Gold-layer feature for:
- care intensity
- chronic disease management
- treatment persistence
- patient complexity scoring

### Expected Output
A new column:

`careplan_duration_days`

In [0]:
careplan_clean_df = careplan_clean_df.withColumn(
    "careplan_duration_days",
    datediff(col("end_datetime"), col("start_datetime"))
)

print("CarePlan duration calculated successfully.")

CarePlan duration calculated successfully.


## Step 8 — Inspect Clean CarePlan Data

### What We Are Doing
We are displaying the clean CarePlan table.

### Why We Are Doing This
We need to verify:
- care plan status
- category
- activity
- patient ID
- encounter ID
- duration

### Expected Output
A clean care plan-level table.

In [0]:
display(careplan_clean_df)

careplan_id,patient_reference,encounter_reference,careplan_status,careplan_intent,careplan_category,activity_description,activity_status,reason_reference,start_datetime,end_datetime,patient_id,encounter_id,reason_condition_id,careplan_duration_days
574fb3a3-f1b9-2d5f-4fae-38a992b3b860,urn:uuid:21dde8d5-5497-8bf3-aa32-b79e998e9024,urn:uuid:d4f072f5-2024-94ee-0aac-5e0d9bae2590,active,order,"{""coding"":[{""system"":""http://hl7.org/fhir/us/core/CodeSystem/careplan-category"",""code"":""assess-plan""}]}",Diabetic diet,in-progress,urn:uuid:a35aae78-6a02-8b06-8c42-f840e60be749,1993-10-03T06:57:45.000Z,null,21dde8d5-5497-8bf3-aa32-b79e998e9024,d4f072f5-2024-94ee-0aac-5e0d9bae2590,a35aae78-6a02-8b06-8c42-f840e60be749,null
da7158be-366b-1c43-2959-9b020be3764c,urn:uuid:21dde8d5-5497-8bf3-aa32-b79e998e9024,urn:uuid:e5186888-3794-a79b-4c45-76dd877dba8b,completed,order,"{""coding"":[{""system"":""http://hl7.org/fhir/us/core/CodeSystem/careplan-category"",""code"":""assess-plan""}]}",Antenatal education,completed,urn:uuid:d81e9f14-3bd6-51e8-f0cc-0b20b69d090f,2013-03-17T06:57:45.000Z,2013-11-03T06:57:45.000Z,21dde8d5-5497-8bf3-aa32-b79e998e9024,e5186888-3794-a79b-4c45-76dd877dba8b,d81e9f14-3bd6-51e8-f0cc-0b20b69d090f,231
d981f82f-27ff-4eb1-2496-1386730c2cdc,urn:uuid:21dde8d5-5497-8bf3-aa32-b79e998e9024,urn:uuid:65efa80c-0faa-52e3-c9a8-9b9efeb75c01,completed,order,"{""coding"":[{""system"":""http://hl7.org/fhir/us/core/CodeSystem/careplan-category"",""code"":""assess-plan""}]}",Antenatal education,completed,urn:uuid:d81e9f14-3bd6-51e8-f0cc-0b20b69d090f,2014-10-05T06:57:45.000Z,2015-05-17T06:57:45.000Z,21dde8d5-5497-8bf3-aa32-b79e998e9024,65efa80c-0faa-52e3-c9a8-9b9efeb75c01,d81e9f14-3bd6-51e8-f0cc-0b20b69d090f,224
3751f42e-a9ce-6044-4244-40a8ca7a57b0,urn:uuid:96a91c58-1099-af08-0aee-3d924cd5c7e8,urn:uuid:7b3c3bc3-5a9e-7ec5-f3ec-5dbe7f359c9b,active,order,"{""coding"":[{""system"":""http://hl7.org/fhir/us/core/CodeSystem/careplan-category"",""code"":""assess-plan""}]}",Food allergy diet,in-progress,null,1984-03-16T03:21:13.000Z,null,96a91c58-1099-af08-0aee-3d924cd5c7e8,7b3c3bc3-5a9e-7ec5-f3ec-5dbe7f359c9b,null,null
514aac99-698e-8806-695b-675a5ccdd000,urn:uuid:96a91c58-1099-af08-0aee-3d924cd5c7e8,urn:uuid:63cad61d-1514-336b-5529-2d545316563f,active,order,"{""coding"":[{""system"":""http://hl7.org/fhir/us/core/CodeSystem/careplan-category"",""code"":""assess-plan""}]}",Inhaled steroid therapy,in-progress,null,1988-04-06T03:25:13.000Z,null,96a91c58-1099-af08-0aee-3d924cd5c7e8,63cad61d-1514-336b-5529-2d545316563f,null,null
3ae6e246-86eb-4b5b-ef3e-805cebbb7722,urn:uuid:96a91c58-1099-af08-0aee-3d924cd5c7e8,urn:uuid:1a1bf36d-2f29-a30e-d040-6cff14da6233,active,order,"{""coding"":[{""system"":""http://hl7.org/fhir/us/core/CodeSystem/careplan-category"",""code"":""assess-plan""}]}",Drug addiction counseling,in-progress,urn:uuid:a4a4416f-cbe1-8db2-2802-171a4c1a44ed,2009-09-09T03:25:13.000Z,null,96a91c58-1099-af08-0aee-3d924cd5c7e8,1a1bf36d-2f29-a30e-d040-6cff14da6233,a4a4416f-cbe1-8db2-2802-171a4c1a44ed,null
b3b3df7c-3fdd-6f84-4091-856674e8b12f,urn:uuid:96a91c58-1099-af08-0aee-3d924cd5c7e8,urn:uuid:6a596836-ceb5-588c-d21b-8d3fda9f0b85,completed,order,"{""coding"":[{""system"":""http://hl7.org/fhir/us/core/CodeSystem/careplan-category"",""code"":""assess-plan""}]}",Dressing change management,completed,urn:uuid:a175f01e-22dd-d45d-a1b7-7dd4912961ae,2013-06-17T03:25:13.000Z,2013-07-06T03:25:13.000Z,96a91c58-1099-af08-0aee-3d924cd5c7e8,6a596836-ceb5-588c-d21b-8d3fda9f0b85,a175f01e-22dd-d45d-a1b7-7dd4912961ae,19
fd645ad5-accb-ce31-a204-2c8fb9238fb0,urn:uuid:96a91c58-1099-af08-0aee-3d924cd5c7e8,urn:uuid:f63b8ce0-7937-88d5-0029-716a6d15980e,completed,order,"{""coding"":[{""system"":""http://hl7.org/fhir/us/core/CodeSystem/careplan-category"",""code"":""assess-plan""}]}",Antenatal education,completed,urn:uuid:0dda4968-efd5-8d85-ff4b-71b2e50b737d,2014-08-16T03:25:13.000Z,2015-03-21T03:25:13.000Z,96a91c58-1099-af08-0aee-3d924cd5c7e8,f63b8ce0-7937-88d5-0029-716a6d1

## Step 9 — Check CarePlan Status Distribution

### What We Are Doing
We are counting care plans by status.

### Why We Are Doing This
This validates whether care plans are active, completed, or cancelled.

### Expected Output
Care plan status frequency table.

In [0]:
display(
    careplan_clean_df.groupBy(
        "careplan_status"
    ).count().orderBy(
        desc("count")
    )
)

careplan_status,count
completed,1065
active,766


## Step 10 — Check Most Common CarePlan Categories

### What We Are Doing
We are counting the most common care plan categories.

### Why We Are Doing This
This helps us understand the clinical care management programs in the dataset.

### Expected Output
CarePlan category frequency table.

In [0]:
display(
    careplan_clean_df.groupBy(
        "careplan_category"
    ).count().orderBy(
        desc("count")
    )
)

careplan_category,count
"{""coding"":[{""system"":""http://hl7.org/fhir/us/core/CodeSystem/careplan-category"",""code"":""assess-plan""}]}",1831


## Step 11 — Check Null Values

### What We Are Doing
We are checking missing values in the clean CarePlan table.

### Why We Are Doing This
Silver-layer quality validation is required before Gold analytics.

### Expected Output
A null-count summary table.

In [0]:
display(
    careplan_clean_df.select(
        [
            sum(col(column_name).isNull().cast("int")).alias(column_name)
            for column_name in careplan_clean_df.columns
        ]
    )
)

careplan_id,patient_reference,encounter_reference,careplan_status,careplan_intent,careplan_category,activity_description,activity_status,reason_reference,start_datetime,end_datetime,patient_id,encounter_id,reason_condition_id,careplan_duration_days
0,0,0,0,0,0,1,1,366,0,766,0,0,366,766


## Step 12 — Save Silver CarePlan Table

### What We Are Doing
We are saving the clean CarePlan table into the Silver layer.

### Why We Are Doing This
This creates a reusable Delta table for analytics and ML.

### Expected Output
A Delta table:

`healthcare_catalog.silver.careplan_clean`

In [0]:
careplan_clean_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("healthcare_catalog.silver.careplan_clean")

print("Silver careplan_clean table saved successfully.")

Silver careplan_clean table saved successfully.


## Step 13 — Verify Silver Tables

### What We Are Doing
We are listing all Silver tables.

### Why We Are Doing This
We want to confirm that `careplan_clean` was saved successfully.

### Expected Output
`careplan_clean` should appear in the Silver table list.

In [0]:
spark.sql("""
SHOW TABLES IN healthcare_catalog.silver
""").show(truncate=False)

+--------+------------------------+-----------+
|database|tableName               |isTemporary|
+--------+------------------------+-----------+
|silver  |careplan_clean          |false      |
|silver  |condition_clean         |false      |
|silver  |encounter_clean         |false      |
|silver  |immunization_clean      |false      |
|silver  |medication_request_clean|false      |
|silver  |observation_clean       |false      |
|silver  |patient_clean           |false      |
|silver  |procedure_clean         |false      |
+--------+------------------------+-----------+

